# MNIST MLP3: SGD + momentum versus SGD + momentum + Spectral RG-Flow

Matched test: both models start from the same weights and receive the same
minibatches. The wrapped arm first completes the ordinary `SGD + momentum` update,
then removes only positive centered-log-spectrum flow along the current
participation-ratio collapse surrogate. This is ordinary torch.optim.SGD with classical momentum=0.9 and nesterov=False; it is not Muon and performs no Newton--Schulz orthogonalization.

The primary test is FC1: does the wrapped run stay closer to $\alpha=2$
without reducing test accuracy?


In [ ]:
import importlib, subprocess, sys
from pathlib import Path

try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "weightwatcher>=0.7.7"]
    )

cwd = Path.cwd().resolve()
package_root = None
for root in [cwd, *cwd.parents]:
    direct = root / "rg_spectral_flow"
    nested = root / "optimizers" / "spectral_rg_flow_projector" / "rg_spectral_flow"
    if direct.is_dir():
        package_root = root
        break
    if nested.is_dir():
        package_root = nested.parent
        break
if package_root is None:
    raise RuntimeError("Run from a clone of CalculatedContent/rg_optimizers.")
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))
print("Package root:", package_root)


In [ ]:
from dataclasses import asdict
import json
import matplotlib.pyplot as plt
import pandas as pd
from rg_spectral_flow.optimizer_benchmarks import (
    OptimizerBenchmarkConfig,
    run_optimizer_benchmark,
)

pd.set_option("display.max_columns", None)
CONFIG = OptimizerBenchmarkConfig(
    optimizer_family="sgd_momentum",
    epochs=20,
    batch_size=128,
    learning_rate=0.05,
    weight_decay=0.0001,
    grad_clip_norm=1.0,
    sgd_momentum=0.9,
    sgd_dampening=0.0,
    sgd_nesterov=False,
    collapse_potential="participation_ratio",
    projection_strength=1.0,
    max_abs_log_eigenvalue_correction=0.20,
    max_correction_ratio=0.10,
    apply_every_steps=25,
    warmup_epochs=1,
    min_retained=20,
    sc_normalization_gamma=0.0,
    sc_support_policy="midpoint",
    ww_svd_method="accurate",
    train_eval_max_batches=50,
)
asdict(CONFIG)

In [ ]:
result = run_optimizer_benchmark(CONFIG, data_dir="./data", progress=True)
RUN_DIR = Path("./runs_mnist_spectral_rg_flow/sgd_momentum")
result.save(RUN_DIR)
(RUN_DIR / "experiment_config.json").write_text(
    json.dumps(asdict(CONFIG), indent=2)
)
print("Saved:", RUN_DIR.resolve())


In [ ]:
# Matched accuracy and layerwise alpha.
display(result.performance.sort_values(["epoch", "flow_enabled"]).tail(12))

fig, ax = plt.subplots(figsize=(9, 5))
for run, frame in result.performance.groupby("run"):
    frame = frame.sort_values("epoch")
    ax.plot(frame["epoch"], frame["test_acc"], marker="o", label=f"{run} test")
    ax.plot(frame["epoch"], frame["train_acc"], linestyle="--", label=f"{run} train")
ax.set(xlabel="Epoch", ylabel="Accuracy", title="SGD + momentum: matched trajectories")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()

valid = result.weightwatcher[result.weightwatcher["status"] == "ok"].copy()
for layer, layer_frame in valid.groupby("layer_name"):
    fig, ax = plt.subplots(figsize=(9, 5))
    for run, frame in layer_frame.groupby("run"):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["alpha"], marker="o", label=run)
    ax.axhline(2.0, linestyle="--", label="alpha = 2")
    ax.set(xlabel="Epoch", ylabel="WeightWatcher alpha", title=f"{layer} alpha")
    ax.grid(True, alpha=0.3); ax.legend(); plt.show()


In [ ]:
# FC1 falsification summary plus implementation diagnostics.
fc1 = valid[
    valid["layer_name"].astype(str).str.contains("fc1", case=False, regex=False)
].sort_values(["run", "epoch"])
if fc1.empty:
    display(valid["layer_name"].drop_duplicates().to_frame())
else:
    spectral = (
        fc1.groupby("run")
        .agg(
            final_alpha=("alpha", "last"),
            minimum_alpha=("alpha", "min"),
            mean_abs_alpha_minus_2=("alpha", lambda x: (x - 2.0).abs().mean()),
            final_erg_gap_sc=("ERG_gap_SC", "last"),
        )
        .reset_index()
    )
    accuracy = (
        result.performance.sort_values("epoch")
        .groupby("run")
        .agg(final_test_acc=("test_acc", "last"), peak_test_acc=("test_acc", "max"))
        .reset_index()
    )
    display(spectral.merge(accuracy, on="run"))

if result.flow_steps.empty:
    print("No spectral-flow evaluations were logged.")
else:
    display(result.correction_summary.tail(30))
    applied = result.flow_steps[result.flow_steps["status"] == "ok"]
    if not applied.empty:
        display(
            applied[
                [
                    "epoch", "global_step", "parameter",
                    "base_flow_component", "corrected_flow_component",
                    "correction_ratio", "correction_capped",
                ]
            ].tail(40)
        )
